In [ ]:
# 6-16-2026

In [34]:
import zarr
import xarray as xr
import numpy as np
import pandas as pd
import fsspec
from dask.diagnostics import ProgressBar


zip_path = "seasfire_v0.4.zip"
inner_path = "seasfire_v0.4.zarr"

In [35]:
store = zarr.storage.ZipStore("seasfire_v0.4.zip", mode="r")
zgroup = zarr.open_group(store=store, path=inner_path, mode="r")
mapper = fsspec.get_mapper("zip://seasfire_v0.4.zarr::seasfire_v0.4.zip")

In [36]:
group = zarr.open_group(store=store, path=inner_path, mode="r")
rows = []
for name in group.array_keys():
    arr = group[name]
    rows.append({"name": name, "shape": arr.shape, "dtype": str(arr.dtype), "chunks": arr.chunks})
inventory = pd.DataFrame(rows).sort_values("name").reset_index(drop=True)

In [37]:
inventory

,name,shape,dtype,chunks
0,area,"(720, 1440)",float32,"(180, 360)"
1,biomes,"(720, 1440)",float32,"(180, 360)"
2,cams_co2fire,"(966, 720, 1440)",float32,"(966, 180, 360)"
3,cams_frpfire,"(966, 720, 1440)",float32,"(966, 180, 360)"
4,drought_code_max,"(966, 720, 1440)",float32,"(966, 180, 360)"
...,...,...,...,...
57,t2m_min,"(966, 720, 1440)",float32,"(966, 180, 360)"
58,time,"(966,)",int64,"(966,)"
59,tp,"(966, 720, 1440)",float32,"(966, 180, 360)"
60,vpd,"(966, 720, 1440)",float32,"(966, 180, 360)"


In [38]:
ds = xr.open_dataset(mapper, engine="zarr", consolidated=True)

In [39]:
list(mapper.keys())

['.zattrs',
 '.zgroup',
 '.zmetadata',
 'area/.zarray',
 'area/.zattrs',
 'area/0.0',
 'area/0.1',
 'area/0.2',
 'area/0.3',
 'area/1.0',
 'area/1.1',
 'area/1.2',
 'area/1.3',
 'area/2.0',
 'area/2.1',
 'area/2.2',
 'area/2.3',
 'area/3.0',
 'area/3.1',
 'area/3.2',
 'area/3.3',
 'biomes/.zarray',
 'biomes/.zattrs',
 'biomes/0.0',
 'biomes/0.1',
 'biomes/0.2',
 'biomes/0.3',
 'biomes/1.0',
 'biomes/1.1',
 'biomes/1.2',
 'biomes/1.3',
 'biomes/2.0',
 'biomes/2.1',
 'biomes/2.2',
 'biomes/2.3',
 'biomes/3.0',
 'biomes/3.1',
 'biomes/3.2',
 'biomes/3.3',
 'cams_co2fire/.zarray',
 'cams_co2fire/.zattrs',
 'cams_co2fire/0.0.0',
 'cams_co2fire/0.0.1',
 'cams_co2fire/0.0.2',
 'cams_co2fire/0.0.3',
 'cams_co2fire/0.1.0',
 'cams_co2fire/0.1.1',
 'cams_co2fire/0.1.2',
 'cams_co2fire/0.1.3',
 'cams_co2fire/0.2.0',
 'cams_co2fire/0.2.1',
 'cams_co2fire/0.2.2',
 'cams_co2fire/0.2.3',
 'cams_co2fire/0.3.0',
 'cams_co2fire/0.3.1',
 'cams_co2fire/0.3.2',
 'cams_co2fire/0.3.3',
 'cams_frpfire/.zarray'

In [40]:
ds

<xarray.Dataset> Size: 164GB
Dimensions:                         (latitude: 720, longitude: 1440, time: 966)
Coordinates:
  * latitude                        (latitude) float64 6kB 89.88 ... -89.88
  * longitude                       (longitude) float64 12kB -179.9 ... 179.9
  * time                            (time) datetime64[ns] 8kB 2001-01-01 ... ...
Data variables: (12/59)
    area                            (latitude, longitude) float32 4MB ...
    biomes                          (latitude, longitude) float32 4MB ...
    cams_co2fire                    (time, latitude, longitude) float32 4GB ...
    cams_frpfire                    (time, latitude, longitude) float32 4GB ...
    drought_code_max                (time, latitude, longitude) float32 4GB ...
    drought_code_mean               (time, latitude, longitude) float32 4GB ...
    ...                              ...
    t2m_max                         (time, latitude, longitude) float32 4GB ...
    t2m_mean                        (time, latitude, longitude) float32 4GB ...
    t2m_min                         (time, latitude, longitude) float32 4GB ...
    tp                              (time, latitude, longitude) float32 4GB ...
    vpd                             (time, latitude, longitude) float32 4GB ...
    ws10                            (time, latitude, longitude) float32 4GB ...
Attributes:
    crs:          EPSG:4326
    description:  The SeasFire Cube is a scientific datacube for seasonal fir...
    title:        SeasFire Cube: A Global Dataset for Seasonal Fire Modeling ...

In [41]:
print(ds.data_vars)

Data variables:
    area                            (latitude, longitude) float32 4MB ...
    biomes                          (latitude, longitude) float32 4MB ...
    cams_co2fire                    (time, latitude, longitude) float32 4GB ...
    cams_frpfire                    (time, latitude, longitude) float32 4GB ...
    drought_code_max                (time, latitude, longitude) float32 4GB ...
    drought_code_mean               (time, latitude, longitude) float32 4GB ...
    fcci_ba                         (time, latitude, longitude) float32 4GB ...
    fcci_ba_valid_mask              (time) int8 966B ...
    fcci_fraction_of_burnable_area  (time, latitude, longitude) float32 4GB ...
    fcci_fraction_of_observed_area  (time, latitude, longitude) float32 4GB ...
    fcci_number_of_patches          (time, latitude, longitude) float32 4GB ...
    fwi_max                         (time, latitude, longitude) float32 4GB ...
    fwi_mean                        (time, latitude, longit

In [42]:
vars_to_remove = ["mslp", "gfed_ba", "gfed_ba_valid_mask", "gfed_region", "oci_ao", "oci_wp", "oci_pna", 
                  "oci_nao", "oci_soi", "oci_gmsst", "oci_pdo", "oci_pdo", "oci_ea", "oci_epo", "oci_nina34_anom", "oci_censo", 
                  "lccs_class_0", "lccs_class_5", "lccs_class_8"]

In [43]:
len(vars_to_remove)

19

In [44]:
ds_lean = ds.drop_vars(vars_to_remove)

In [45]:
ds_filtered = ds_lean.sel(time=slice("2011-01-01", "2021-12-31"))

In [46]:
ds_chunked = ds_filtered.chunk({"time": -1, "latitude": 45, "longitude": 45})

In [47]:
for var in ds_chunked.variables:
    if "chunks" in ds_chunked[var].encoding:
        del ds_chunked[var].encoding["chunks"]

In [48]:
delayed_export = ds_chunked.to_zarr(
    "seasfire_filtered.zarr", 
    mode="w", 
    consolidated=True, 
    compute=False
)

In [ ]:
with ProgressBar(): # takes ~ 60 min
    delayed_export.compute(scheduler="sync")

[########################################] | 100% Completed | 61m 55s
